# NeurIPS Diffusion Evaluation

Train diffusion model, compute FID (AE or GyroSwin latent space), evaluate warm restarts, test FID-vs-convergence.

In [1]:
%load_ext autoreload
%autoreload 2

import sys, os
sys.path.append("..")
os.environ["CUDA_VISIBLE_DEVICES"] = "2"

In [ ]:
import omegaconf, yaml
from collections import defaultdict

import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from scipy.stats import pearsonr
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

from neugk.diffusion import get_diffusion_runner

from neurips_diff_eval import (
    flux_uq_for_valset, plot_flux_confidence,
    compute_statistics, compute_fid,
    compute_fid_on_latents, extract_gyroswin_latents,
    to_model_space, from_model_space,
    run_trajectory_pair, time_to_convergence,
    plot_correlation_grid, load_reference_flux
)

## 0. Configuration

In [ ]:
DATA_PATH = "/local00/bioinf/galletti/preprocessed_kvikio"
# AE_CHECKPOINT = "/restricteddata/ukaea/checkpoints/autoencoders/neurips26/simsiam_normal/20260328_172956_442/best.pth"
AE_CHECKPOINT = "/restricteddata/ukaea/checkpoints/autoencoders/neurips26/ae_small/20260328_115627_555/best.pth"
GYROSWIN_CHECKPOINT = "/system/user/publicwork/galletti/checkpoints/gyroswin_large/pytorch_model.bin"
GKW_RAW_DIR = "/restricteddata/ukaea/gyrokinetics/raw"

ID_VAL = ["iteration_262.h5", "iteration_135.h5", "iteration_8.h5",
          "iteration_232.h5", "iteration_148.h5", "iteration_115.h5"]
OOD_VAL = [f"ood_iteration_{i}.h5" for i in range(5)]
TRAIN_TRAJS = "iteration_{0-5,7-12,14-31,33-82,84-99}.h5"

N_EPOCHS = 100
BATCH_SIZE = 512
LR = 1e-3
VAL_EVERY = 10
N_DENOISING_STEPS = 10

# Diffusion training options
MINIBATCH_OT = True
NOISE_DISTRIBUTION = "gaussian"  # "gaussian" or "mixture"
CONTINUOUS_TIME = True

FID_MODE = "gyroswin"  # "ae" or "gyroswin"
FID_N_COMPONENTS = 512
FID_MAX_SAMPLES = 128  # None for full

## 1. Train Diffusion Model

In [ ]:
cfg = omegaconf.OmegaConf.load(os.path.join(os.path.dirname(AE_CHECKPOINT), "config.yaml"))
with open("dit_config.yaml", "r") as f:
    diff_cfg = omegaconf.DictConfig(yaml.safe_load(f))
cfg.model = diff_cfg.model
cfg.ddp = diff_cfg.ddp
cfg.workflow = "diffusion"

cfg.output_path = "/tmp/diffusion_eval_pipeline"
cfg.dataset.path = DATA_PATH
cfg.dataset.gds_override = False  # switch to true if precompute latents
cfg.dataset.backend = "gds"
cfg.ae_checkpoint = os.path.dirname(AE_CHECKPOINT)
cfg.dataset.training_trajectories = TRAIN_TRAJS
cfg.dataset.validation_trajectories = ID_VAL
cfg.model.latent_dim = 512
cfg.training.batch_size = BATCH_SIZE
cfg.training.learning_rate = LR
cfg.training.n_epochs = N_EPOCHS
cfg.validation.validate_every_n_epochs = VAL_EVERY
cfg.validation.probe = {"targets": []}
cfg.logging.writer = None
cfg.logging.tqdm = True
cfg.training.num_workers = 0
cfg.training.pin_memory = False  # in-memory latents, no worker overhead
cfg.model.diffusion.formulation = "edm"
cfg.model.diffusion.minibatch_ot = MINIBATCH_OT
cfg.model.diffusion.noise_distribution = NOISE_DISTRIBUTION
cfg.model.diffusion.continuous_time = CONTINUOUS_TIME
cfg.dataset.normalization = {
    "df": {"type": "zscore", "agg_axes": [0, 1, 3, 4, 5]},
    "flux": {"type": "zscore", "agg_axes": None}, 
    "phi": {"type": "zscore", "agg_axes": [0, 1, 2]}
}

runner = get_diffusion_runner(rank=0, cfg=cfg, world_size=1)
print(f"Model: {sum(p.numel() for p in runner.model.parameters())/1e6:.1f}M params")
print(f"Train: {len(runner.trainset)}, Val: {sum(len(v) for v in runner.valsets)}")

In [ ]:
raw = runner.trainset.__getitem__(100, get_normalized=False, override_latens=True)       
print(f"raw std: {raw.df.std():.4f}, raw mean: {raw.df.mean():.4f}")                     
scale, shift = runner.trainset._get_scale_shift(0, "df", raw.df)                         
print(f"shift shape: {shift.shape}, scale shape: {scale.shape}")                         
print(f"shift: {shift.squeeze()}")                                                       
print(f"scale: {scale.squeeze()}")                                                       
manual = (raw.df - shift) / scale                                                      
print(f"manual normalized std: {manual.std():.4f}") 

In [ ]:
from neugk.plot_utils import plot_nd

sample = runner.trainset.__getitem__(100, get_normalized=True, override_latens=True)
df_in = sample.df.unsqueeze(0).to(runner.device)
cond = sample.conditioning.unsqueeze(0).to(runner.device)

ae = runner.autoencoder
ae.eval()
with torch.no_grad():
    recon = ae(df_in, condition=cond)["df"]

df_gt = df_in[0].cpu()
df_rec = recon[0].cpu()

print(f"Input shape: {df_gt.shape}, Recon shape: {df_rec.shape}")
print(f"AE recon MSE: {(df_gt - df_rec).pow(2).mean():.6f}")
print(f"AE recon rel err: {(df_gt - df_rec).norm() / df_gt.norm():.4f}")

_ = plot_nd(df_gt, df_rec, to_wandb=False)

In [ ]:
logs = runner(skip_eval=True)

In [ ]:
for k, v in logs[-1].items():
    if "info/" in str(k):
        print(f"  {k}: {v:.1f}")

In [ ]:
sample = runner.trainset.__getitem__(0, get_normalized=True, override_latens=True)
cond = sample.conditioning.unsqueeze(0).to(runner.device)

runner.model.eval()
with torch.no_grad():
    gen_out = runner.sample(cond, latent_only=False, steps=N_DENOISING_STEPS)

df_gt = sample.df.cpu()
df_gen = gen_out["df"][0].cpu()

print(f"GT shape: {df_gt.shape}, Gen shape: {df_gen.shape}")
print(f"Gen vs GT rel err: {(df_gt - df_gen).norm() / df_gt.norm():.4f}")

_ = plot_nd(df_gt, df_gen, to_wandb=False)

In [ ]:
epochs = [l["epoch"] for l in logs]
train_loss = [l.get("train/loss", l.get("train/df", np.nan)) for l in logs]
val_epochs = [l["epoch"] for l in logs if "val_traj/avg_flux_rmse" in l]
val_rmse = [l["val_traj/avg_flux_rmse"] for l in logs if "val_traj/avg_flux_rmse" in l]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(epochs, train_loss, lw=1)
ax1.set(xlabel="Epoch", ylabel="Loss", title="Training loss", yscale="log")
ax1.grid(True, alpha=0.15)
if val_rmse:
    ax2.plot(val_epochs, val_rmse, "o-", markersize=4)
ax2.set(xlabel="Epoch", ylabel="Avg flux RMSE", title="Validation")
ax2.grid(True, alpha=0.15)
fig.tight_layout()

for l in reversed(logs):
    if "val_plots" in l:
        for name, obj in l["val_plots"].items():
            if hasattr(obj, "savefig"): display(obj)
        break

## 1.5 Flux UQ (ID)

In [ ]:
id_ids, id_means, id_stds, id_gts = flux_uq_for_valset(
    runner, runner.valsets[0], n_samples_per_traj=32, steps=N_DENOISING_STEPS)
plot_flux_confidence(id_ids, id_means, id_stds, id_gts, title="Flux UQ - ID")
print(f"ID flux RMSE: {np.sqrt(((id_means - id_gts)**2).mean()):.4f}")

## 2. FID Evaluation

- **`ae`**: FID in AE bottleneck (fast, no decode)
- **`gyroswin`**: FID using frozen GyroSwin encoder (Inception-FID analog)

In [ ]:
#  investigate if models are compilable, check the speedup. if they are not, make them so,
#   then check the speedup. focus on all layers @neugk/models/ ,
#   @neugk/pinc/autoencoders/gk_autoencoders.py basic autoencoder (not vqvae or vae).
#   ENSURE OUTPUTS STAY CONSISTENT

In [ ]:
if FID_MODE == "gyroswin":
    sd = torch.load(GYROSWIN_CHECKPOINT, map_location="cpu", weights_only=False)

    with open("../configs/model/multi.yaml") as f:
        gs_cfg = omegaconf.DictConfig(yaml.safe_load(f))

    from neugk.gyroswin.models.gyroswin import GyroSwinMultitask
    from neugk.models.layers import ContinuousConditionEmbed

    # infer dim and in_channels from checkpoint weights
    ckpt_dim = sd["df_unet.down_blocks.0.swin_att.blocks.0.attn.proj.bias"].shape[0]
    ckpt_in_channels = sd["df_unet.vel_pe.pos_embed"].shape[-1]
    ckpt_real_potens = sd["phi_unet.unpatch.expansion.bias"].shape[0] == 1

    df_base_resolution = list(runner.trainset.resolution)
    phi_base_resolution = list(runner.trainset.phi_resolution)
    n_cond = len(gs_cfg.conditioning)

    cond_fn = ContinuousConditionEmbed(128, n_cond)
    flux_cond_fn = ContinuousConditionEmbed(128, n_cond) if gs_cfg.swin.flux_conditioning else None
    outputs = [k for k in gs_cfg.loss_weights if gs_cfg.loss_weights[k] > 0.0 or gs_cfg.loss_scheduler[k]]

    print(f"Checkpoint: dim={ckpt_dim}, in_channels={ckpt_in_channels}, real_potens={ckpt_real_potens}")

    gyroswin_model = GyroSwinMultitask(
        dim=ckpt_dim,
        outputs=outputs,
        df_base_resolution=df_base_resolution,
        phi_base_resolution=phi_base_resolution,
        df_patch_size=gs_cfg.swin.patch_size,
        phi_patch_size=gs_cfg.swin.phi_patch_size,
        df_window_size=gs_cfg.swin.window_size,
        phi_window_size=gs_cfg.swin.phi_window_size,
        depth=gs_cfg.swin.depth,
        num_heads=gs_cfg.swin.num_heads,
        in_channels=ckpt_in_channels,
        out_channels=ckpt_in_channels,
        num_layers=gs_cfg.num_layers,
        use_checkpoint=gs_cfg.swin.gradient_checkpoint,
        drop_path=gs_cfg.swin.drop_path,
        use_abs_pe=gs_cfg.swin.use_abs_pe,
        c_multiplier=gs_cfg.swin.c_multiplier,
        modulation=gs_cfg.swin.modulation,
        merging_hidden_ratio=gs_cfg.swin.merging_hidden_ratio,
        unmerging_hidden_ratio=gs_cfg.swin.unmerging_hidden_ratio,
        act_fn=getattr(torch.nn, gs_cfg.swin.act_fn),
        patch_skip=gs_cfg.swin.patch_skip,
        decouple_mu=gs_cfg.decouple_mu,
        swin_bottleneck=gs_cfg.swin.swin_bottleneck,
        use_rpb=gs_cfg.swin.use_rpb,
        use_rope=gs_cfg.swin.use_rope,
        latent_cross_attn=gs_cfg.swin.latent_cross_attn,
        real_potens=ckpt_real_potens,
        flux_reduce=gs_cfg.swin.flux_reduce,
        flux_num_heads=gs_cfg.swin.flux_num_heads,
        flux_depth=gs_cfg.swin.flux_depth,
        cond_embed=cond_fn,
        flux_cond_embed=flux_cond_fn,
        conditioning=list(gs_cfg.conditioning),
        init_weights=gs_cfg.swin.init_weights,
        patching_init_weights=gs_cfg.swin.patching_init_weights,
        cond_init_weights=gs_cfg.swin.cond_init_weights,
    )
    info = gyroswin_model.load_state_dict(sd, strict=False)
    if info.unexpected_keys:
        print(f"Unexpected: {info.unexpected_keys[:5]}...")
    if info.missing_keys:
        print(f"Missing: {info.missing_keys[:5]}...")
    gyroswin_model.eval().to(runner.device)
    print(f"GyroSwin: {sum(p.numel() for p in gyroswin_model.parameters())/1e6:.1f}M")
    feature_fn = lambda b, device, **kw: extract_gyroswin_latents(gyroswin_model, b, device, **kw)
    fid_label = "GyroSwin-FID"
else:
    feature_fn = None
    fid_label = "AE-FID"
print(f"Mode: {fid_label}")

In [ ]:
GEN_BATCH_SIZE = 32
runner.model.eval()

if FID_MODE == "ae":
    real_lats, real_conds = [], []
    for (fi, ti), s in tqdm(runner.trainset.precomputed_latents.items(), desc="Real latents"):
        real_lats.append(np.array(s["x"]).reshape(-1))
        real_conds.append(np.array([s["itg"], s["dg"], s["s_hat"], s["q"]]))
        if FID_MAX_SAMPLES and len(real_lats) >= FID_MAX_SAMPLES: break
    X_real, C_real = np.stack(real_lats), np.stack(real_conds)
    idx = np.random.choice(len(X_real), len(X_real), replace=True)
    gen_lats = []
    for i in tqdm(range(0, len(X_real), GEN_BATCH_SIZE), desc="Gen latents"):
        c = torch.tensor(C_real[idx[i:i+GEN_BATCH_SIZE]], dtype=torch.float32, device=runner.device)
        with torch.no_grad():
            z = runner.sample(c, latent_only=True, steps=N_DENOISING_STEPS)
            gen_lats.append(z.cpu().numpy().reshape(z.shape[0], -1))
    X_gen = np.concatenate(gen_lats)

elif FID_MODE == "gyroswin":
    n = FID_MAX_SAMPLES or min(512, len(runner.valsets[0]))
    real_s = [runner.valsets[0][i].df for i in tqdm(range(n), desc="Loading real")]
    gen_s = []
    # build conditioning from metadata (avoid slow valset.__getitem__)
    _cond_meta_map = {"itg": "ion_temp_grad", "dg": "density_grad"}
    _cond_keys = sorted(runner.cfg.model.conditioning)
    _val_conds = []
    for idx in range(n):
        fi, _ = runner.valsets[0].flat_index_to_file_and_tstep[idx]
        meta = runner.valsets[0].metadata[fi]
        _val_conds.append(torch.tensor([float(np.squeeze(meta[_cond_meta_map.get(k, k)])) for k in _cond_keys], dtype=torch.float32))
    conds = torch.stack(_val_conds)
    for i in tqdm(range(0, n, GEN_BATCH_SIZE), desc="Generating"):
        c = conds[i:i+GEN_BATCH_SIZE].to(runner.device)
        with torch.no_grad():
            p = runner.sample(c, steps=N_DENOISING_STEPS, latent_only=False)
            gen_s.extend([p["df"][j].cpu() for j in range(p["df"].shape[0])])
    print(f"Extracting features...")
    _, X_real, X_gen = compute_fid_on_latents(
        real_s, gen_s, feature_fn, device=runner.device, n_components=FID_N_COMPONENTS)

print(f"Real: {X_real.shape}, Gen: {X_gen.shape}")

In [ ]:
X_rf, X_gf = X_real, X_gen
if FID_N_COMPONENTS and FID_N_COMPONENTS < X_real.shape[1]:
    pca = PCA(n_components=FID_N_COMPONENTS)
    X_rf = pca.fit_transform(X_real)
    X_gf = pca.transform(X_gen)
    print(f"PCA: {X_real.shape[1]}->{FID_N_COMPONENTS} ({pca.explained_variance_ratio_.sum():.1%} var)")

mu_r, sig_r = compute_statistics(X_rf)
mu_g, sig_g = compute_statistics(X_gf)
fid_global = compute_fid(mu_r, sig_r, mu_g, sig_g)
print(f"Global {fid_label}: {fid_global:.4f}")

In [ ]:
per_traj_fids = {}
if FID_MODE == "ae":
    ptr = defaultdict(list)
    for (fi, ti), s in runner.trainset.precomputed_latents.items():
        ptr[fi].append(np.array(s["x"]).reshape(-1))
    for fi, rl in tqdm(ptr.items(), desc="Per-traj FID"):
        Xt = np.stack(rl)
        if len(Xt) < 10: continue
        sk = next(k for k in runner.trainset.precomputed_latents if k[0] == fi)
        s = runner.trainset.precomputed_latents[sk]
        c = torch.tensor([[s["itg"], s["dg"], s["s_hat"], s["q"]]],
                          dtype=torch.float32, device=runner.device).expand(len(Xt), -1)
        with torch.no_grad():
            zg = runner.sample(c, latent_only=True, steps=N_DENOISING_STEPS)
            Xg = zg.cpu().numpy().reshape(zg.shape[0], -1)
        if FID_N_COMPONENTS and FID_N_COMPONENTS < Xt.shape[1]:
            Xt, Xg = pca.transform(Xt), pca.transform(Xg)
        m1, s1 = compute_statistics(Xt)
        m2, s2 = compute_statistics(Xg)
        per_traj_fids[fi] = compute_fid(m1, s1, m2, s2)
    print(f"Per-traj mean: {np.mean(list(per_traj_fids.values())):.4f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
mr, mg = X_real.mean(0), X_gen.mean(0)
axes[0].scatter(mr, mg, alpha=0.3, s=10, color="#264653")
lim = [min(mr.min(), mg.min()), max(mr.max(), mg.max())]
axes[0].plot(lim, lim, "r--", alpha=0.8)
axes[0].set(title="Feature means", xlabel="Real", ylabel="Gen")
sr, sg = X_real.std(0), X_gen.std(0)
axes[1].scatter(sr, sg, alpha=0.3, s=10, color="#e9c46a")
lim = [min(sr.min(), sg.min()), max(sr.max(), sg.max())]
axes[1].plot(lim, lim, "r--", alpha=0.8)
axes[1].set(title="Feature stds", xlabel="Real", ylabel="Gen")
if per_traj_fids:
    axes[2].hist(list(per_traj_fids.values()), bins=30, color="#2a9d8f", edgecolor="k", alpha=0.8)
    axes[2].axvline(fid_global, color="red", ls="--", lw=1.5, label=f"Global={fid_global:.1f}")
    axes[2].set(title=f"Per-traj {fid_label}", xlabel="FID")
    axes[2].legend()
else:
    axes[2].text(0.5, 0.5, "N/A", ha="center", va="center", transform=axes[2].transAxes)
fig.suptitle(f"{fid_label} = {fid_global:.2f}", fontweight="bold")
fig.tight_layout()

## 3. Warm Restart Evaluation

In [ ]:
from gyaradax import gk_from_gkw_dir

In [ ]:
N_EVAL_STEPS = 1000
WARM_RESTART_ITERATIONS = [13]

warm_results = {}
_cond_keys = sorted(runner.cfg.model.conditioning)

for iteration in WARM_RESTART_ITERATIONS:
    print(f"\n{'=' * 88}\nIteration {iteration}\n{'=' * 88}")
    df_gt, geometry, params, state_init, pre = gk_from_gkw_dir(
        os.path.join(GKW_RAW_DIR, f"iteration_{iteration}"), mixed_precision=True), k_index=80

    # build conditioning from GKW params
    _param_map = {"itg": "rlt", "dg": "rln", "s_hat": "shat", "q": "q"}
    cond = torch.tensor(
        [[float(getattr(params, _param_map[k])) for k in _cond_keys]],
        dtype=torch.float32, device=runner.device,
    )

    matching = [f for f in runner.trainset.files if f"iteration_{iteration}" in f]
    fi = runner.trainset.files.index(matching[0]) if matching else -1

    with torch.no_grad():
        DF_PRED = runner.sample(cond, latent_only=False, steps=N_DENOISING_STEPS)["df"][0].cpu().numpy()

    df_warm = from_model_space(DF_PRED)
    log_gt, log_warm = run_trajectory_pair(
        df_gt, df_warm, geometry, params, pre, state_init,
        N_EVAL_STEPS, f"iter-{iteration}", chunk_size=10,
        backend="jax", print_every=100,
    )
    ref_mean, ref_std = load_reference_flux(GKW_RAW_DIR, iteration)
    print(f"  Reference flux: mean={ref_mean:.3e}, std={ref_std:.3e}")
    ttc = time_to_convergence(log_warm, log_gt, log_gt["time"],
                              ref_flux_mean=ref_mean, ref_flux_std=ref_std)
    ttc_flux, ttc_spec = ttc["flux"], ttc["ky_spec"]
    fid_val = per_traj_fids.get(fi, fid_global if 'fid_global' in dir() else np.nan)

    warm_results[iteration] = dict(
        fid=fid_val, ttc_flux=ttc_flux, ttc_spec=ttc_spec, log_gt=log_gt, log_warm=log_warm, df_pred=DF_PRED,
    )
    print(f"  TTC_flux={ttc_flux:.3f}, TTC_spec={ttc_spec:.3f}, FID={fid_val:.4f}")

In [ ]:
# 5D visualization: generated IC vs final state after warm-start simulation
from neugk.plot_utils import plot_nd

for it, res in warm_results.items():
    # generated initial condition (model space, 4ch with separate_zf)
    df_ic = torch.tensor(res["df_pred"], dtype=torch.float32)

    # final state of warm-started simulation (spectral -> model space)
    df_final = torch.tensor(to_model_space(res["log_warm"]["df_final"]), dtype=torch.float32)

    print(f"\nIteration {it}: IC vs final warm-start state")
    fig = plot_nd(df_ic, df_final, to_wandb=False)
    fig.suptitle(f"Iter {it}: Generated IC (left) vs Warm-start final (right)",
                 fontsize=11, y=1.01)
    plt.show()

In [ ]:
n = len(warm_results)
fig, axes = plt.subplots(n, 3, figsize=(15, 4*n), squeeze=False)
snap_colors = ["#2a9d8f", "#e76f51", "#264653"]

for row, (it, res) in enumerate(warm_results.items()):
    lg, lw, t = res["log_gt"], res["log_warm"], res["log_gt"]["time"]

    # ky spectrum: compare at 3 snapshots (early, mid, late)
    n_t = len(t)
    snap_idx = [min(5, n_t-1), n_t//2, n_t-1]
    for j, si in enumerate(snap_idx):
        ky_gt = np.log10(np.maximum(lg["ky_spec"][si], 1e-30))
        ky_w  = np.log10(np.maximum(lw["ky_spec"][si], 1e-30))
        lbl = f"t={float(t[si]):.1f}"
        axes[row,0].plot(ky_gt, "-",  color=snap_colors[j], lw=1.2, alpha=0.7, label=f"GT {lbl}")
        axes[row,0].plot(ky_w,  "--", color=snap_colors[j], lw=1.2, alpha=0.9, label=f"warm {lbl}")
    axes[row,0].set(title=f"iter {it}: $k_y$ spectrum", xlabel="$k_y$ mode", ylabel="log$_{10}$(E)")
    axes[row,0].legend(fontsize=6, ncol=2); axes[row,0].grid(True, alpha=0.15)

    # flux time series with reference band
    ref_mean, ref_std = load_reference_flux(GKW_RAW_DIR, it)
    axes[row,1].plot(t, lg["eflux"], "k", lw=0.8, alpha=0.5, label="GT")
    axes[row,1].plot(t, lw["eflux"], lw=1, label="warm")
    axes[row,1].axhspan(ref_mean - ref_std, ref_mean + ref_std, color="k", alpha=0.08, label=f"ref ±1σ")
    axes[row,1].axhline(ref_mean, color="k", ls=":", lw=0.8)
    if res["ttc_flux"] < np.inf:
        axes[row,1].axvline(t[0]+res["ttc_flux"], color="green", ls="-", lw=1.5, label=f"TTC_flux={res['ttc_flux']:.1f}")
    if res["ttc_spec"] < np.inf:
        axes[row,1].axvline(t[0]+res["ttc_spec"], color="blue", ls="--", lw=1.5, label=f"TTC_spec={res['ttc_spec']:.1f}")
    axes[row,1].set_title(f"iter {it}: flux"); axes[row,1].legend(fontsize=6); axes[row,1].grid(True, alpha=0.15)

    # Pearson(ky) over time
    n_compare = min(len(lw["ky_spec"]), len(lg["ky_spec"]))
    r_ts = [pearsonr(np.log10(np.maximum(lw["ky_spec"][i],1e-30)),
                      np.log10(np.maximum(lg["ky_spec"][i],1e-30)))[0]
            for i in range(n_compare)]
    axes[row,2].plot(t[:n_compare], r_ts, lw=1)
    axes[row,2].axhline(0.95, color="gray", ls="--", lw=0.8, label="0.95")
    if res["ttc_spec"] < np.inf:
        axes[row,2].axvline(t[0]+res["ttc_spec"], color="blue", ls="--", lw=1, alpha=0.5)
    axes[row,2].set_title(f"iter {it}: Pearson($k_y$)"); axes[row,2].set_ylim(-0.1,1.05)
    axes[row,2].legend(fontsize=7); axes[row,2].grid(True, alpha=0.15)

for ax in axes[-1]: ax.set_xlabel(r"time $[v_{th}/R]$")
fig.tight_layout()

## 4. FID vs. Time-to-Convergence

In [ ]:
fv, tv, lb = [], [], []
for it, res in warm_results.items():
    if np.isfinite(res["fid"]) and np.isfinite(res["ttc_flux"]):
        fv.append(res["fid"]); tv.append(res["ttc_flux"]); lb.append(f"iter_{it}")
fv, tv = np.array(fv), np.array(tv)

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(fv, tv, s=80, zorder=3, edgecolors="k", linewidth=0.5)
for i, l in enumerate(lb): ax.annotate(l, (fv[i], tv[i]), textcoords="offset points", xytext=(8,4), fontsize=9)
if len(fv)>=3:
    co = np.polyfit(fv, tv, 1)
    xf = np.linspace(fv.min(), fv.max(), 100)
    ax.plot(xf, np.polyval(co, xf), "r--", lw=1.5)
    r, p = pearsonr(fv, tv)
    ax.text(0.05, 0.95, f"r={r:.3f}, p={p:.3f}", transform=ax.transAxes, fontsize=12, va="top",
            bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5))
ax.set(xlabel=fid_label, ylabel=r"TTC $[v_{th}/R]$", title=f"{fid_label} vs TTC")
ax.grid(True, alpha=0.15)
fig.tight_layout()

In [ ]:
print(f"{'Iter':>8} {'FID':>10} {'TTC_flux':>10} {'TTC_spec':>10} {'RelErr':>10}")
print("-"*54)
for it, r in warm_results.items():
    f = f"{r['fid']:.4f}" if np.isfinite(r['fid']) else 'N/A'
    tf = f"{r['ttc_flux']:.3f}" if np.isfinite(r['ttc_flux']) else 'inf'
    ts = f"{r['ttc_spec']:.3f}" if np.isfinite(r['ttc_spec']) else 'inf'

## 5. Extended Correlation Analysis

In [ ]:
extended_metrics = {}
for it, res in warm_results.items():
    m = dict(fid=res["fid"], ttc=res["ttc_flux"])
    lg, lw = res["log_gt"], res["log_warm"]
    gm = np.mean(lg["eflux"][-80:])
    m["integral_flux_err"] = float(abs(lw["eflux"][0] - gm) / max(abs(gm), 1e-30))
    ky_lg = np.log10(np.maximum(np.mean(lg["ky_spec"][-80:], 0), 1e-30))
    ky_lw = np.log10(np.maximum(lw["ky_spec"][0], 1e-30))
    r_ky, _ = pearsonr(ky_lw, ky_lg) if len(ky_lg)>1 else (0.,1.)
    m["kyspec_pearson_init"] = float(r_ky)
    m["kyspec_rmse_init"] = float(np.sqrt(np.mean((ky_lw - ky_lg)**2)))
    kx_lg = np.log10(np.maximum(np.mean(lg["kx_spec"][-80:], 0), 1e-30))
    kx_lw = np.log10(np.maximum(lw["kx_spec"][0], 1e-30))
    m["kxspec_rmse_init"] = float(np.sqrt(np.mean((kx_lw - kx_lg)**2)))
    m["probe_flux_rmse"] = np.nan  # needs flux head
    extended_metrics[it] = m
display(pd.DataFrame(extended_metrics).T.round(4))

In [ ]:
_ = plot_correlation_grid(extended_metrics)